In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/branch_sql_MVP/settings.json').is_file())
sys.path.insert(0, str(WORKSPACE)) if str(WORKSPACE) not in sys.path else None
MVP_ROOT = WORKSPACE / 'src/branch_sql_MVP'
os.chdir(MVP_ROOT)


# Luna — tối ưu trực tiếp trên test (8 kịch bản)

Theo yêu cầu người dùng: ngân sách mới tối đa **9.000.000 token**, dùng 99/307 câu internal test, phủ 11 database. Screening dùng 33 câu nằm trong 99 câu này. Các tham số được chọn bằng điểm test, nên mọi kết quả ở đây là **test-tuned / in-sample**, không phải đánh giá độc lập. 208 câu còn lại không thuộc lượt chạy này.

Gold SQL và kết quả gold chỉ đi vào evaluator, không vào prompt sinh SQL, selector hoặc repair. Khởi tạo từ cấu hình đã dev trước đó; chạy đủ 8 cấu hình gốc, thử một biến thể cho mỗi kịch bản trên 33 câu, mở rộng biến thể triển vọng lên 99 câu nếu ngân sách cho phép. Chỉ so sánh/chọn cuối cùng các lượt hoàn tất trên cùng 99 câu. Đây là tìm kiếm có giới hạn, không chứng minh tối ưu toàn cục. Mỗi cấu hình chỉ có một lượt sinh; chưa đo phương sai qua nhiều lần chạy hay kiểm định ý nghĩa thống kê. Seed 42 cố định việc chọn mẫu, không đảm bảo đầu ra LLM tái lập tuyệt đối.

Notebook chỉ đọc kết quả, không gọi API. Runner: `src/eval/run_test_optimization.py`. Ledger mới độc lập với ledger ngày trước; các reservation chưa xác nhận vẫn được tính vào trần. ID giữ tiền tố `vi:dev` của dữ liệu nguồn; vai trò internal test được xác định bằng `src/eval/dataset_split.json`, không phải tên thư mục nguồn. G1 nhận difficulty metadata từ dataset trong bước định tuyến; đây không phải chế độ question-only. Các kịch bản sử dụng evidence được cung cấp của benchmark.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
root = MVP_ROOT
runtime = root / '.runtime/luna_test_optimization_20260908'
bundle = json.loads((runtime / 'bundle.json').read_text(encoding='utf-8'))
display(pd.Series(bundle['protocol'], name='Protocol').to_frame())
print('Status:', bundle['status'], '| Accounted:', f"{bundle['accounted_tokens']:,}")
if bundle.get('stop_reason'): print(bundle['stop_reason'])

,Protocol
evaluation_role,test_tuned_in_sample
independent_test,False
test_cases,99
screen_cases,33
remaining_unrun,208
seed,42
token_limit,9000000
model,gpt-5.6-luna
concurrency,6
selection,screen alternative on 33; extend promising alt...


Status: complete_budgeted_search | Accounted: 8,706,990


# Kịch bản và thông số

P1: cung cấp toàn bộ context. B4-hybrid: retrieval hybrid. Event-seeds: chỉ lấy event hạt giống. Event-expansion: mở rộng event. B5-selector: thêm contextual selector. B6-relational: context quan hệ + vòng sửa SQL. B6-hybrid: retrieval hybrid + vòng sửa SQL. G1-adaptive: định tuyến sinh ứng viên SQL. Tên hiển thị phân biệt các ablation dù cùng dùng một graph nền.

Mỗi biến thể chỉ thay nhóm tham số ghi trong `alternative`; mọi tham số còn lại giữ nguyên. Model sinh: `gpt-5.6-luna`, max output 8192, reasoning high trừ biến thể P1 medium. Luật chọn: execution accuracy cao hơn, rồi ít run error, ít SQL không hợp lệ, ít input/output token hơn.

In [2]:
display(pd.json_normalize(bundle['specs']).T)
from src.branch_sql_MVP.data.catalog import load_benchmark_cases
from src.branch_sql_MVP.settings import load_settings
settings = load_settings()
lookup = {c.stable_id: c for c in load_benchmark_cases(settings.path(settings.paths.dev), 'dev')}
samples = pd.DataFrame([{'stable_id': key, 'database': lookup[key].db_id, 'difficulty': lookup[key].difficulty, 'screen': key in bundle['screen_ids']} for key in bundle['test_ids']])
display(pd.crosstab(samples.database, samples.difficulty))
display(samples)

,0,1,2,3,4,5,6,7
name,P1-full,B4-hybrid,B4-event-seeds,B4-event-expansion,B5-selector,B6-relational,B6-hybrid,G1-adaptive
scenario,P1,B4,B5,B5,B5,B6,B6,G1
alternative.reasoning,medium,NaN,NaN,NaN,NaN,NaN,NaN,NaN
context.mode,NaN,hybrid,hybrid,hybrid,hybrid,hybrid,hybrid,hybrid
context.candidate_k,NaN,16.0,16.0,16.0,16.0,16.0,16.0,16.0
context.docs_top_k,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0
context.semantic_weight,NaN,0.7,0.7,0.7,0.7,0.7,0.7,0.7
context.keyword_weight,NaN,0.3,0.3,0.3,0.3,0.3,0.3,0.3
context.rrf_k,NaN,40.0,40.0,40.0,40.0,40.0,40.0,40.0
context.rerank_top_k,NaN,8.0,8.0,8.0,8.0,8.0,8.0,8.0


difficulty,challenging,moderate,simple
database,,,
california_schools,1,3,3
card_games,3,3,5
codebase_community,1,3,4
debit_card_specializing,1,3,3
european_football_2,3,4,4
financial,1,3,3
formula_1,3,3,3
student_club,2,3,4
superhero,3,4,3


,stable_id,database,difficulty,screen
0,vi:dev:california_schools:31,california_schools,moderate,False
1,vi:dev:california_schools:32,california_schools,moderate,False
2,vi:dev:california_schools:36,california_schools,challenging,True
3,vi:dev:california_schools:44,california_schools,simple,False
4,vi:dev:california_schools:49,california_schools,moderate,True
...,...,...,...,...
94,vi:dev:toxicology:231,toxicology,challenging,True
95,vi:dev:toxicology:233,toxicology,simple,True
96,vi:dev:toxicology:284,toxicology,moderate,False
97,vi:dev:toxicology:313,toxicology,simple,False


# Kết quả gốc — cùng 99 câu

Execution accuracy loại các trường hợp gold execution lỗi khỏi mẫu số; coverage được báo riêng. Không dùng kết quả screening 33 câu thay thế kết quả 99 câu.

In [3]:
def table(rows):
    return pd.json_normalize(rows)
pd.set_option('display.max_columns', None)
display(table(bundle['baseline']))

,name,run_id,scenario,config.name,config.scenario,manifest.run_id,manifest.scenario,manifest.split,manifest.model_provider,manifest.model,manifest.prompt_version,manifest.dataset_fingerprint,manifest.index_fingerprint,manifest.dependency_lock_sha256,manifest.seed,manifest.hardware.platform,manifest.hardware.python,manifest.hardware.torch,manifest.hardware.cuda_build,manifest.hardware.cuda_available,manifest.hardware.device,manifest.parameters.context.evaluation_role,manifest.parameters.execution.timeout_seconds,manifest.parameters.execution.max_rows,manifest.parameters.execution.eval_timeout_seconds,manifest.parameters.max_repairs,manifest.parameters.max_concurrency,manifest.parameters.generation.system_prompt,manifest.parameters.generation.temperature,manifest.parameters.generation.max_tokens,manifest.parameters.generation.top_p,manifest.parameters.generation.stop,manifest.parameters.generation.structured_output,manifest.parameters.generation.extra.reasoning_effort,manifest.parameters.example_corpus,manifest.parameters.evaluation_protocol,manifest.parameters.evaluation_error_policy,manifest.created_at,metrics.cases,metrics.completed_cases,metrics.execution_accuracy,metrics.evaluable_cases,metrics.evaluation_error_cases,metrics.evaluation_coverage,metrics.exact_sql_match,metrics.invalid_sql_rate,metrics.empty_result_rate,metrics.run_error_rate,metrics.mean_generation_latency_seconds,metrics.input_tokens,metrics.output_tokens,metrics.mean_repairs,metrics.mean_candidates,metrics.r_ves,metrics.r_ves_reason,metrics.input_tokens_all_calls,metrics.output_tokens_all_calls,config.context.mode,config.context.candidate_k,config.context.docs_top_k,config.context.semantic_weight,config.context.keyword_weight,config.context.rrf_k,config.context.rerank_top_k,config.context.rerank_enabled,config.context.rerank_max_length,config.context.rerank_batch_size,config.context.table_k,config.context.column_k,config.context.schema_min_score,config.context.value_k,config.context.value_fuzzy_threshold,config.context.token_budget,config.context.event_top_k,config.context.event_hops,config.context.event_node_budget,config.context.event_token_budget,config.context.selector_max_items,config.context.example_k,config.context.use_events,config.context.expand_events,manifest.parameters.context.mode,manifest.parameters.context.candidate_k,manifest.parameters.context.docs_top_k,manifest.parameters.context.semantic_weight,manifest.parameters.context.keyword_weight,manifest.parameters.context.rrf_k,manifest.parameters.context.rerank_top_k,manifest.parameters.context.rerank_enabled,manifest.parameters.context.rerank_max_length,manifest.parameters.context.rerank_batch_size,manifest.parameters.context.table_k,manifest.parameters.context.column_k,manifest.parameters.context.schema_min_score,manifest.parameters.context.value_k,manifest.parameters.context.value_fuzzy_threshold,manifest.parameters.context.token_budget,manifest.parameters.context.event_top_k,manifest.parameters.context.event_hops,manifest.parameters.context.event_node_budget,manifest.parameters.context.event_token_budget,manifest.parameters.context.selector_max_items,manifest.parameters.context.example_k,manifest.parameters.context.use_events,manifest.parameters.context.expand_events,config.context.contextual_selector,manifest.parameters.context.contextual_selector,config.context.use_relational_context,config.max_repairs,manifest.parameters.context.use_relational_context,config.route.max_candidates,manifest.parameters.route.max_candidates
0,P1-full,test-tuned-ac877a542175,P1,P1-full,P1,test-tuned-ac877a542175,P1,dev,openai,gpt-5.6-luna,bird-sql-v1,f177dc09b400b66e48df250cb3643a5e5039b13fbdc088...,3600417e2ef9d29421869a16b46c1417a37664edeacda1...,40483697b00ca0eb2b03d2fbfcfefbf22914d1da5e0de0...,42,Windows-11-10.0.26200-SP0,3.12.13,2.11.0+cu128,12.8,True,NVIDIA GeForce RTX 4050 Laptop GPU,test_tuned_in_sample,5.0,500,10.0,0,6,Bạn là trợ lý Text-to-SQL. Chỉ dùng ngữ cảnh đ...,None,8192,None,[],None,high,unav

# Thử cấu hình thay thế trên 33 câu và mở rộng lên 99

`delta` là thay đổi accuracy so với cấu hình gốc trên đúng 33 câu screening. `skipped=budget` nghĩa là không đủ ngân sách dự kiến để bắt đầu; không phải kết quả kém. Mọi lượt dở dang vẫn nằm trong `runs/*/cases.jsonl`, có thể resume và không được tính thành lượt so sánh hoàn tất.

In [4]:
display(table(bundle['screening']))
display(table(bundle['extensions']))

,name,run_id,scenario,delta,config.name,config.scenario,config.reasoning,manifest.run_id,manifest.scenario,manifest.split,manifest.model_provider,manifest.model,manifest.prompt_version,manifest.dataset_fingerprint,manifest.index_fingerprint,manifest.dependency_lock_sha256,manifest.seed,manifest.hardware.platform,manifest.hardware.python,manifest.hardware.torch,manifest.hardware.cuda_build,manifest.hardware.cuda_available,manifest.hardware.device,manifest.parameters.context.evaluation_role,manifest.parameters.execution.timeout_seconds,manifest.parameters.execution.max_rows,manifest.parameters.execution.eval_timeout_seconds,manifest.parameters.max_repairs,manifest.parameters.max_concurrency,manifest.parameters.generation.system_prompt,manifest.parameters.generation.temperature,manifest.parameters.generation.max_tokens,manifest.parameters.generation.top_p,manifest.parameters.generation.stop,manifest.parameters.generation.structured_output,manifest.parameters.generation.extra.reasoning_effort,manifest.parameters.example_corpus,manifest.parameters.evaluation_protocol,manifest.parameters.evaluation_error_policy,manifest.created_at,metrics.cases,metrics.completed_cases,metrics.execution_accuracy,metrics.evaluable_cases,metrics.evaluation_error_cases,metrics.evaluation_coverage,metrics.exact_sql_match,metrics.invalid_sql_rate,metrics.empty_result_rate,metrics.run_error_rate,metrics.mean_generation_latency_seconds,metrics.input_tokens,metrics.output_tokens,metrics.mean_repairs,metrics.mean_candidates,metrics.r_ves,metrics.r_ves_reason,metrics.input_tokens_all_calls,metrics.output_tokens_all_calls,baseline_metrics.cases,baseline_metrics.completed_cases,baseline_metrics.execution_accuracy,baseline_metrics.evaluable_cases,baseline_metrics.evaluation_error_cases,baseline_metrics.evaluation_coverage,baseline_metrics.exact_sql_match,baseline_metrics.invalid_sql_rate,baseline_metrics.empty_result_rate,baseline_metrics.run_error_rate,baseline_metrics.mean_generation_latency_seconds,baseline_metrics.input_tokens,baseline_metrics.output_tokens,baseline_metrics.mean_repairs,baseline_metrics.mean_candidates,baseline_metrics.r_ves,baseline_metrics.r_ves_reason,baseline_metrics.input_tokens_all_calls,baseline_metrics.output_tokens_all_calls,config.context.mode,config.context.candidate_k,config.context.docs_top_k,config.context.semantic_weight,config.context.keyword_weight,config.context.rrf_k,config.context.rerank_top_k,config.context.rerank_enabled,config.context.rerank_max_length,config.context.rerank_batch_size,config.context.table_k,config.context.column_k,config.context.schema_min_score,config.context.value_k,config.context.value_fuzzy_threshold,config.context.token_budget,config.context.event_top_k,config.context.event_hops,config.context.event_node_budget,config.context.event_token_budget,config.context.selector_max_items,config.context.example_k,config.context.use_events,config.context.expand_events,manifest.parameters.context.mode,manifest.parameters.context.candidate_k,manifest.parameters.context.docs_top_k,manifest.parameters.context.semantic_weight,manifest.parameters.context.keyword_weight,manifest.parameters.context.rrf_k,manifest.parameters.context.rerank_top_k,manifest.parameters.context.rerank_enabled,manifest.parameters.context.rerank_max_length,manifest.parameters.context.rerank_batch_size,manifest.parameters.context.table_k,manifest.parameters.context.column_k,manifest.parameters.context.schema_min_score,manifest.parameters.context.value_k,manifest.parameters.context.value_fuzzy_threshold,manifest.parameters.context.token_budget,manifest.parameters.context.event_top_k,manifest.parameters.context.event_hops,manifest.parameters.context.event_node_budget,manifest.parameters.context.event_token_budget,manifest.parameters.context.selector_max_items,manifest.parameters.context.example_k,manifest.parameters.context.use_events,manifest.parameters.context.expand_events,config.context.contextual_selector,manifest.parameters.context.context

,name,run_id,scenario,config.name,config.scenario,config.context.mode,config.context.candidate_k,config.context.docs_top_k,config.context.semantic_weight,config.context.keyword_weight,config.context.rrf_k,config.context.rerank_top_k,config.context.rerank_enabled,config.context.rerank_max_length,config.context.rerank_batch_size,config.context.table_k,config.context.column_k,config.context.schema_min_score,config.context.value_k,config.context.value_fuzzy_threshold,config.context.token_budget,config.context.event_top_k,config.context.event_hops,config.context.event_node_budget,config.context.event_token_budget,config.context.selector_max_items,config.context.example_k,config.context.use_events,config.context.expand_events,config.context.contextual_selector,manifest.run_id,manifest.scenario,manifest.split,manifest.model_provider,manifest.model,manifest.prompt_version,manifest.dataset_fingerprint,manifest.index_fingerprint,manifest.dependency_lock_sha256,manifest.seed,manifest.hardware.platform,manifest.hardware.python,manifest.hardware.torch,manifest.hardware.cuda_build,manifest.hardware.cuda_available,manifest.hardware.device,manifest.parameters.context.mode,manifest.parameters.context.candidate_k,manifest.parameters.context.docs_top_k,manifest.parameters.context.semantic_weight,manifest.parameters.context.keyword_weight,manifest.parameters.context.rrf_k,manifest.parameters.context.rerank_top_k,manifest.parameters.context.rerank_enabled,manifest.parameters.context.rerank_max_length,manifest.parameters.context.rerank_batch_size,manifest.parameters.context.table_k,manifest.parameters.context.column_k,manifest.parameters.context.schema_min_score,manifest.parameters.context.value_k,manifest.parameters.context.value_fuzzy_threshold,manifest.parameters.context.token_budget,manifest.parameters.context.event_top_k,manifest.parameters.context.event_hops,manifest.parameters.context.event_node_budget,manifest.parameters.context.event_token_budget,manifest.parameters.context.selector_max_items,manifest.parameters.context.example_k,manifest.parameters.context.use_events,manifest.parameters.context.expand_events,manifest.parameters.context.contextual_selector,manifest.parameters.context.evaluation_role,manifest.parameters.execution.timeout_seconds,manifest.parameters.execution.max_rows,manifest.parameters.execution.eval_timeout_seconds,manifest.parameters.max_repairs,manifest.parameters.max_concurrency,manifest.parameters.generation.system_prompt,manifest.parameters.generation.temperature,manifest.parameters.generation.max_tokens,manifest.parameters.generation.top_p,manifest.parameters.generation.stop,manifest.parameters.generation.structured_output,manifest.parameters.generation.extra.reasoning_effort,manifest.parameters.example_corpus,manifest.parameters.evaluation_protocol,manifest.parameters.evaluation_error_policy,manifest.created_at,metrics.cases,metrics.completed_cases,metrics.execution_accuracy,metrics.evaluable_cases,metrics.evaluation_error_cases,metrics.evaluation_coverage,metrics.exact_sql_match,metrics.invalid_sql_rate,metrics.empty_result_rate,metrics.run_error_rate,metrics.mean_generation_latency_seconds,metrics.input_tokens,metrics.output_tokens,metrics.mean_repairs,metrics.mean_candidates,metrics.r_ves,metrics.r_ves_reason,metrics.input_tokens_all_calls,metrics.output_tokens_all_calls,config.context.use_relational_context,config.max_repairs,manifest.parameters.context.use_relational_context
0,B4-event-seeds-alternative,test-tuned-89f7785807d2,B5,B4-event-seeds-alternative,B5,hybrid,16,5,0.7,0.3,40,8,True,256,8,8,20,0.12,8,0.84,5000,4,1,24,1800,24,0,True,False,False,test-tuned-89f7785807d2,B5,dev,openai,gpt-5.6-luna,bird-sql-v1,f177dc09b400b66e48df250cb3643a5e5039b13fbdc088...,3600417e2ef9d29421869a16b46c1417a37664edeacda1...,40483697b00ca0eb2b03d2fbfcfefbf22914d1da5e0de0...,42,Windows-11-10.0.26200-SP0,3.12.13,2.11.0+cu128,12.8,True,NVIDIA GeForce RTX 4050 Laptop GPU,hybrid,16,5,0.7,0.3,40,8,True,256,8,8,20,0.12,8,0.84,5000,4

# Cấu hình được chọn — kết quả test-tuned

Chỉ những cấu hình hoàn tất 99 câu mới đủ điều kiện được chọn. Chọn dựa trên chính test nên có selection bias; cần một tập chưa dùng để tuning nếu muốn ước lượng tổng quát hóa.

In [5]:
selected = table(bundle['selected'])
columns = ['name', 'scenario_name', 'run_id', 'metrics.execution_accuracy', 'metrics.evaluable_cases', 'metrics.evaluation_coverage', 'metrics.run_error_rate', 'metrics.invalid_sql_rate', 'metrics.input_tokens_all_calls', 'metrics.output_tokens_all_calls']
display(selected[[c for c in columns if c in selected]])
display(selected.T)

,name,scenario_name,run_id,metrics.execution_accuracy,metrics.evaluable_cases,metrics.evaluation_coverage,metrics.run_error_rate,metrics.invalid_sql_rate,metrics.input_tokens_all_calls,metrics.output_tokens_all_calls
0,P1-full,NaN,test-tuned-ac877a542175,0.581633,98,0.989899,0.0,0.020202,883705,36446
1,B4-hybrid,NaN,test-tuned-86ac98e088d7,0.551020,98,0.989899,0.0,0.010101,422576,39842
2,B4-event-seeds-alternative,B4-event-seeds,test-tuned-89f7785807d2,0.581633,98,0.989899,0.0,0.020202,533106,40708
3,B4-event-expansion,NaN,test-tuned-fac59443886e,0.561224,98,0.989899,0.0,0.030303,566050,37142
4,B5-selector-alternative,B5-selector,test-tuned-cc7d5a3bdbd2,0.571429,98,0.989899,0.0,0.030303,641047,92591
5,B6-relational-alternative,B6-relational,test-tuned-793d197b5a8a,0.561224,98,0.989899,0.0,0.010101,644498,95529
6,B6-hybrid-alternative,B6-hybrid,test-tuned-d853384e9076,0.571429,98,0.989899,0.0,0.010101,433315,40467
7,G1-adaptive,NaN,test-tuned-a1236d2ecf45,0.551020,98,0.989899,0.0,0.000000,600870,52887


,0,1,2,3,4,5,6,7
name,P1-full,B4-hybrid,B4-event-seeds-alternative,B4-event-expansion,B5-selector-alternative,B6-relational-alternative,B6-hybrid-alternative,G1-adaptive
run_id,test-tuned-ac877a542175,test-tuned-86ac98e088d7,test-tuned-89f7785807d2,test-tuned-fac59443886e,test-tuned-cc7d5a3bdbd2,test-tuned-793d197b5a8a,test-tuned-d853384e9076,test-tuned-a1236d2ecf45
scenario,P1,B4,B5,B5,B5,B6,B6,G1
config.name,P1-full,B4-hybrid,B4-event-seeds-alternative,B4-event-expansion,B5-selector-alternative,B6-relational-alternative,B6-hybrid-alternative,G1-adaptive
config.scenario,P1,B4,B5,B5,B5,B6,B6,G1
...,...,...,...,...,...,...,...,...
config.context.use_relational_context,NaN,NaN,NaN,NaN,NaN,True,False,NaN
config.max_repairs,NaN,NaN,NaN,NaN,NaN,2.0,2.0,NaN
manifest.parameters.context.use_relational_context,NaN,NaN,NaN,NaN,NaN,True,False,NaN
config.route.max_candidates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


# Token và kết quả từng câu

Accounted bao gồm reservation chưa xác nhận. Confirmed lấy từ usage API; token output đã bao gồm reasoning nếu API tính chung. Đây là số token tiêu thụ của lượt chạy, không xác nhận quota miễn phí hay số tiền được miễn.

In [6]:
entries = [json.loads(line) for line in (runtime / 'token_ledger.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()] if (runtime / 'token_ledger.jsonl').exists() else []
confirmed_in = sum(e.get('usage', {}).get('input_tokens', 0) for e in entries)
confirmed_out = sum(e.get('usage', {}).get('output_tokens', 0) for e in entries)
accounted = sum(e['delta'] for e in entries)
display(pd.Series({'limit': 9_000_000, 'confirmed_input': confirmed_in, 'confirmed_output': confirmed_out, 'accounted': accounted, 'uncertain_or_inflight': accounted-confirmed_in-confirmed_out, 'remaining': 9_000_000-accounted}).to_frame('tokens'))
assert accounted <= 9_000_000
records = []
for chosen in bundle['selected']:
    latest = {}
    for line in (runtime / 'runs' / chosen['run_id'] / 'cases.jsonl').read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        if not row.get('run_error'): latest[row['stable_id']] = row
    for key in bundle['test_ids']:
        if key in latest: records.append({'experiment': chosen.get('scenario_name', chosen['name']), **latest[key]})
display(pd.json_normalize(records))

,tokens
limit,9000000
confirmed_input,7927768
confirmed_output,779222
accounted,8706990
uncertain_or_inflight,0
remaining,293010


,experiment,stable_id,scenario,db_id,difficulty,route,repair_count,candidates,observations,evidence_ids,trajectory,total_input_tokens,total_output_tokens,execution_correct,evaluation_error,exact_sql_match,prediction_status,prediction_error,prediction_rows,gold_status,order_matters,evaluation_protocol,prediction.sql,prediction.confidence,prediction.assumptions,prediction.model,prediction.prompt_version,prediction.latency_seconds,prediction.input_tokens,prediction.output_tokens
0,P1-full,vi:dev:california_schools:31,P1,california_schools,moderate,None,0,"[{'candidate_id': 'candidate_1', 'strategy': '...",[],"[full:california_schools:schema, full:californ...","[{'node': 'assemble_full_context', 'estimated_...",8329,1175,True,None,False,success,None,2,success,True,sqlite_result_equivalence_v1,SELECT `Free Meal Count (K-12)` / `Enrollment ...,0.96,[Tỷ lệ được tính là `Free Meal Count (K-12)` /...,gpt-5.6-luna,bird-sql-v1,16.300848,8329,1175
1,P1-full,vi:dev:california_schools:32,P1,california_schools,moderate,None,0,"[{'candidate_id': 'candidate_1', 'strategy': '...",[],"[full:california_schools:schema, full:californ...","[{'node': 'assemble_full_context', 'estimated_...",8343,664,False,None,False,success,None,5,success,True,sqlite_result_equivalence_v1,"SELECT f.""School Name"", f.""FRPM Count (K-12)"" ...",0.97,[],gpt-5.6-luna,bird-sql-v1,9.628076,8343,664
2,P1-full,vi:dev:california_schools:36,P1,california_schools,challenging,None,0,"[{'candidate_id': 'candidate_1', 'strategy': '...",[],"[full:california_schools:schema, full:californ...","[{'node': 'assemble_full_context', 'estimated_...",8317,1249,False,None,False,success,None,1,success,True,sqlite_result_equivalence_v1,"WITH top_schools AS (SELECT s.AdmFName1, s.Adm...",0.98,[Nếu có nhiều trường đồng hạng về số lượng thí...,gpt-5.6-luna,bird-sql-v1,16.454606,8317,1249
3,P1-full,vi:dev:california_schools:44,P1,california_schools,simple,None,0,"[{'candidate_id': 'candidate_1', 'strategy': '...",[],"[full:california_schools:schema, full:californ...","[{'node': 'assemble_full_context', 'estimated_...",8280,209,True,None,False,success,None,1,success,True,sqlite_result_equivalence_v1,"SELECT s.AvgScrWrite, sch.City FROM satscores ...",0.99,[],gpt-5.6-luna,bird-sql-v1,4.647550,8280,209
4,P1-full,vi:dev:california_schools:49,P1,california_schools,moderate,None,0,"[{'candidate_id': 'candidate_1', 'strategy': '...",[],"[full:california_schools:schema, full:californ...","[{'node': 'assemble_full_context', 'estimated_...",8286,592,False,None,False,success,None,879,success,True,sqlite_result_equivalence_v1,"WITH county_closed_counts AS (SELECT County, C...",0.97,[Xác định trường đã đóng cửa bằng schools.Stat...,gpt-5.6-luna,bird-sql-v1,9.843224,8286,592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,G1-adaptive,vi:dev:toxicology:231,G1,toxicology,challenging,sag,0,"[{'candidate_id': 'candidate_1', 'strategy': '...","[{'status': 'success', 'error': None, 'columns...",[schema:toxicology:fk:atom:molecule_id:molecul...,"[{'node': 'choose_context_route', 'route': 'sa...",4893,747,False,None,False,success,None,1,success,True,sqlite_result_equivalence_v1,"WITH bond_counts AS (SELECT b.bond_type, COUNT...",0.98,[Nếu có nhiều loại liên kết đồng hạng về số lư...,gpt-5.6-luna,bird-sql-v1,6.814283,4720,575
788,G1-adaptive,vi:dev:toxicology:233,G1,toxicology,simple,full,0,"[{'candidate_id': 'candidate_1', 'strategy': '...","[{'status': 'success', 'error': None, 'columns...","[full:toxicology:schema, full:toxicology:busin...","[{'node': 'choose_context_route', 'route': 'fu...",7089,492,True,None,False,success,None,2,success,True,sqlite_result_equivalence_v1,SELECT bond_id FROM bond WHERE molecule_id = '...,0.98,[“Liên kết” được hiểu là mã liên kết (bond_id)...,gpt-5.6-luna,bird-sql-v1,3.593200,6925,272
789,G1-adaptive,vi:dev:toxicology:284,G1,toxicology,moderate,full,0,"[{'candidate_id': 'candidate_1', 'strategy': '...","[